## Homework 4


### Preparation

In [4]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [5]:
type(documents[0])

dict

In [6]:
print(len(documents))

72


In [7]:
type(documents[0])

dict

In [8]:
!echo $PREFIX

$PREFIX


In [9]:
#PREFIX="https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main"


In [10]:
#!wget ${PREFIX}../01-agentic-rag/code/rag_helper.py


In [11]:
#!wget ${PREFIX}/04-evaluation/code/evaluation_utils.py

In [12]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

## Q1. Generating questions
Generating questions for all 72 pages costs money and takes time, so let's start small and generate questions for just the first 3 pages:
  
01-agentic-rag/lessons/01-intro.md  
01-agentic-rag/lessons/02-environment.md  
01-agentic-rag/lessons/03-rag.md  

Each call returns the token usage, which most LLM APIs report on the response object (e.g. response.usage.input_tokens / prompt_tokens).  

   

In [13]:
for i in documents[:3]:
    print(i['filename'])

01-agentic-rag/lessons/01-intro.md
01-agentic-rag/lessons/02-environment.md
01-agentic-rag/lessons/03-rag.md


In [14]:
doc=documents[0]
import json
user_prompt = json.dumps(doc)

In [15]:
from evaluation_utils import calc_price, llm_structured_retry, calc_total_price

In [16]:
from dotenv import load_dotenv
from openai import OpenAI
import os

load_dotenv()

# use this for chatgpt
openai_client=OpenAI()
model="gpt-5.4-mini"

In [17]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [18]:
out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )


In [19]:
usage.input_tokens

1020

In [20]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "filename": doc["filename"]
        })

    return results, usage

In [21]:
generate_ground_truth(doc)

([{'question': 'What exactly is a RAG system, and how does it help when an LLM doesn’t know the answer on its own?',
   'filename': '01-agentic-rag/lessons/01-intro.md'},
  {'question': 'Why does this course treat LLMs like black boxes instead of explaining how they work inside?',
   'filename': '01-agentic-rag/lessons/01-intro.md'},
  {'question': 'What are the main limits of LLMs that make retrieval useful, like outdated knowledge or not seeing my files?',
   'filename': '01-agentic-rag/lessons/01-intro.md'},
  {'question': 'What is the FAQ agent in this module supposed to do, and what kind of questions will it answer?',
   'filename': '01-agentic-rag/lessons/01-intro.md'},
  {'question': 'What will be built in the first part of the module, and how is that different from the more agent-like version later on?',
   'filename': '01-agentic-rag/lessons/01-intro.md'}],
 ResponseUsage(input_tokens=1020, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_

In [22]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:3]):
    records, usage = generate_ground_truth(doc)
    print(usage)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/3 [00:00<?, ?it/s]

ResponseUsage(input_tokens=1020, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=114, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1134)
ResponseUsage(input_tokens=1286, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=114, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1400)
ResponseUsage(input_tokens=1753, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=115, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1868)


In [23]:
len(usages)

3

In [24]:
for usage in usages:
    inputtokens=[]
    inputtokens.append(usage.input_tokens)
mean = sum(inputtokens) / len(inputtokens)
print(mean)


1753.0


What's the average number of input tokens across these 3 calls?  

140  
**1400**  
14000  
140000

## Q2. First result with text search  


In [32]:
import pandas as pd
df=pd.read_csv('ground-truth.csv')

In [33]:
ground_truth = df.to_dict(orient="records")

In [34]:
len(ground_truth)

360

We search over the same chunks as in homework 2.

Create them with chunk_documents:

In [35]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
len(chunks)

295

This gives 295 chunks.  

Now rebuild the search from homework 2 over these chunks. Build a text index (Index) and a vector index (VectorSearch), both keyed on filename. Wrap each one in a function, text_search and vector_search, that takes a query and the number of results to return (5 by default).  
  
For hybrid search, reuse the rrf function from homework 2:  

In [97]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            #key = (doc["filename"], doc["start"])
            key = (doc["filename"], doc["content"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

Then define hybrid_search on top of it:

In [98]:
def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

In [38]:
# Take the first question from the ground truth:
q = ground_truth[0]["question"]
q

"What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?"

In [47]:
from minsearch import VectorSearch, Index
from embedder import Embedder
import numpy as np
em=Embedder()


In [48]:
# Extract the text field from each dictionary
text_list = [chunk['content'] for chunk in chunks]
doc_vectors = em.encode_batch(text_list)

In [49]:
index = Index(
    text_fields=['content'],
    
)
index.fit(documents)

In [50]:
X=np.array(doc_vectors)
X.shape

(295, 384)

In [51]:
# Vector search
vindex = VectorSearch(keyword_fields=['content'])
vindex.fit(X, chunks)

In [52]:
def text_search(query, num_results=5):
        boost_dict = {'content':1.0}
       

        return index.search(
            query,
            num_results=num_results,
            boost_dict=boost_dict
            
        )

In [60]:
def vector_search(query, num_results=5):
        boost_dict = {'content':1.0}
        vector_query=em.encode(query)
       

        return vindex.search(
            vector_query,
            num_results=num_results,
            
            
        )

In [54]:
text_result=text_search(q)

In [55]:
text_result[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

In [56]:
text_search(q)[0]['filename']

'01-agentic-rag/lessons/01-intro.md'

### After running text_search for it, what's the filename of the first result?  

**01-agentic-rag/lessons/01-intro.md**  
01-agentic-rag/lessons/03-rag.md  
01-agentic-rag/lessons/13-function-calling.md  
01-agentic-rag/lessons/10-rag-next-steps.md  

In [57]:
v1=em.encode(q)
results=vindex.search(v1, num_results=5)
results[0]['filename']

'01-agentic-rag/lessons/01-intro.md'

In [61]:
results=vector_search(q, num_results=5)
results[0]['filename']

'01-agentic-rag/lessons/01-intro.md'

## Q3. First result with vector search
After running vector_search for the same question, what's the filename of the first result?  

**01-agentic-rag/lessons/01-intro.md**  
01-agentic-rag/lessons/03-rag.md  
04-evaluation/lessons/11-evaluation-intro.md  
04-evaluation/lessons/12-rag-answers.md    

In [62]:
def compute_relevance_text(q):
    doc_id = q["filename"]
    results = text_search(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == doc_id))

    return relevance

In [64]:
from tqdm.auto import tqdm

def compute_relevance_total_text(ground_truth):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance_text(q)
        relevance_total.append(relevance)

    return relevance_total

In [67]:
def compute_relevance_vector(q):
    doc_id = q["filename"]
    question=q['question']
    v1=em.encode(question)
    results=vindex.search(v1, num_results=5)
    

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == doc_id))

    return relevance

In [68]:
from tqdm.auto import tqdm

def compute_relevance_total_vector(ground_truth):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance_vector(q)
        relevance_total.append(relevance)

    return relevance_total

In [69]:
relevance = compute_relevance_total_text(ground_truth)

  0%|          | 0/360 [00:00<?, ?it/s]

In [70]:
relevanceV = compute_relevance_total_vector(ground_truth)
#relevanceV

  0%|          | 0/360 [00:00<?, ?it/s]

In [71]:
relevance[0]

[1, 0, 0, 0, 0]

In [73]:
relevanceV[0]

[1, 0, 0, 0, 0]

In [74]:
def compute_relevance(q, search_function):
    doc_id = q["filename"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == doc_id))

    return relevance

In [75]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [82]:
relevance_total = compute_relevance_total(ground_truth, text_search)

  0%|          | 0/360 [00:00<?, ?it/s]

In [83]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

In [84]:
hit_rate(relevance_total)

0.7555555555555555

In [85]:
relevance_total = compute_relevance_total(ground_truth, vector_search)

  0%|          | 0/360 [00:00<?, ?it/s]

In [86]:
hit_rate(relevance_total)

0.725

In [88]:
mrr(relevanceV)

0.5486111111111112

## Q4. Evaluating text search  
Evaluate text_search on the ground truth data.  

What's the Hit Rate?  

0.55  
0.66  
**0.76**  
0.88  



In [89]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                score = 1 / (rank + 1)
                total_score = total_score + score
                break

    return total_score / len(relevance)

In [90]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [91]:
evaluate(ground_truth, vector_search)

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.725, 'mrr': 0.5486111111111112}

## Q5. Evaluating vector search  
Now evaluate vector_search - the part we left for the homework, since the module only evaluated keyword search.  

What's the MRR?  

0.35  
0.45  
**0.55**  
0.65  

In [100]:
k_values = [1, 50, 100, 200]

results = {}

for k in k_values:
    relevance = []

    for item in ground_truth:
        query = item["question"]
        expected_doc = item["filename"]

        search_results = hybrid_search(query, k=k)

        # check if returned documents match the expected document
        relevance_line = []

        for doc in search_results:
            if doc["filename"] == expected_doc:
                relevance_line.append(1)
            else:
                relevance_line.append(0)

        relevance.append(relevance_line)

    score = mrr(relevance)
    results[k] = score

results

{1: 0.6270370370370372,
 50: 0.5922222222222223,
 100: 0.5922222222222223,
 200: 0.5922222222222223}

## Q6. Tuning hybrid search  
The k constant in RRF controls how much the top ranks matter. A smaller k sharpens the gap between positions, so being at the top of a list counts for more. The RRF paper uses 60 as a default, but the best value depends on the data

so let's measure it.  
Evaluate hybrid_search over the full ground truth dataset for k values 1, 50, 100, and 200. Compare the MRR values for these runs.

Which k gives the best MRR?  

**1**  
50  
100  
200  
Several values of k may give the same MRR. If there's a tie, pick the smallest k.